**Tutorial: 2022 Flood Impact in Pakistan — Remote Sensing Assessment of Agricultural and Urban Damage**

*Author identities and affiliations withheld for double-blind review.*

This tutorial demonstrates a complete, hands-on workflow for assessing flood impacts using remote sensing data and machine learning techniques.

By the end of this tutorial, participants will be able to:

* Detect flood-affected areas using Sentinel-1 SAR imagery

* Classify and quantify damage to crops and built-up areas using Landsat-9 and ESA WorldCover data

* Analyze soil moisture changes using NASA SMAP datasets

* Visualize and interpret results interactively in Python with Google Earth Engine (GEE)


This tutorial is based on the methodology and results from a prior peer-reviewed publication: 2022 Flood Impact in Pakistan: Remote Sensing Assessment of Agricultural and Urban Damage, AAAI Symposium 2024.
**A link to the original publication will be provided upon acceptance.**

# Table of Contents


*   [1. Climate Impact & Motivation](#climate-impact)
*   [2. Target Audience](#target-audience)
*   [3. Background & Prerequisites](#background-prereqs)
*   [4. Software & Data Requirements](#software-requirements)
*   [6. Study Area & Event Description](#data-description)
*   [7. Methodology Overview](#methodology)
*   [8. Hands-On Implementation in Python](#implementation)
*   [9. Results & Discussion](#results-and-discussion)
*   [10. References](#references)




<a name="climate-impact"></a>
# Climate Impact & Motivation

Flooding is a recurrent disaster in Pakistan, particularly during the monsoon season (July–September). Historical records from 1950–2011 show:

- 21 major flood events

- 8,887 fatalities

- Destruction of over 100,000 villages

- Economic losses exceeding $19 billion

The **2022 flood** was unprecedented in scale:
- **$14.9 billion** in damages (The World Bank 2022).

- **$15.2 billion** in economic losses (The World Bank 2022).

- Severe destruction of agricultural land and urban areas, especially in Sindh, Balochistan, and southern Punjab.

Floods not only destroy crops in the immediate aftermath but also affect soil moisture levels, delaying subsequent planting seasons and reducing yields. The paper shows that:

- Kharif crops (planted March–November) were heavily damaged.

- High soil moisture levels after the flood delayed Rabi crop cultivation.

**Why remote sensing?**
Remote sensing and AI-based analysis allow:

-  Fastest source of flood mappingt ((Haq et al. 2012), (Ulloa et al. 2022), (Zhang and Xia 2021)).

- Cost-effective, large-scale damage assessment.

- Better planning for disaster recovery and climate resilience.

This tutorial bridges scientific research with practical implementation, enabling you to replicate and adapt this workflow for other regions and disasters.



# Target Audience

This tutorial is intended for:
- **Researchers** and **students** in environmental science, remote sensing, GIS, and data science.
- **Disaster management professionals** who need to rapidly assess flood impacts.
- **Policy planners and NGOs** working in climate adaptation and food security.

**Prerequisites**:
- Basic Python programming skills.
- Familiarity with geospatial data (raster and vector formats).
- Understanding of basic remote sensing indices like NDVI, EVI, and NDWI.

No prior experience with Google Earth Engine (GEE) is required — this notebook uses the Python API, and all steps are explained in detail.


# Background & Prerequisites
<a name="background-prereqs"></a>

**Remote Sensing in Flood Assessment**
Remote sensing enables large-scale, rapid, and repeatable monitoring of floods. Various sensors provide complementary capabilities:
- **Optical imagery** (Landsat, Sentinel-2) for vegetation and land cover analysis.
- **Synthetic Aperture Radar (SAR)** (Sentinel-1) for detecting water under cloudy conditions.

**Spectral Indices**
Key vegetation and water indices used in flood analysis:
- **NDWI** (Normalized Difference Water Index): Detects water bodies and flood extent.
- **NDVI** (Normalized Difference Vegetation Index): Measures vegetation health.
- **EVI** (Enhanced Vegetation Index): Improves sensitivity in high-biomass areas.

**Machine Learning for Classification**
Supervised classifiers like Random Forest can distinguish between crop types or detect changes before and after flood events.

**Before You Begin**
- Install Python 3.8+ with packages: `geemap`, `earthengine-api`, `folium`, `numpy`, `matplotlib`.
- Create a [Google Earth Engine](https://earthengine.google.com/) account and authenticate it for Python.


<a name="software-requirements"></a>
# Software & Data Requirements

**Python Packages**
- `geemap` – Interactive geospatial mapping in Python.
- `earthengine-api` – Access Google Earth Engine datasets and processing.
- `folium` – Web-based map visualization.
- `numpy`, `matplotlib` – Data analysis and plotting.

**Satellite Datasets**
1. **Sentinel-1 SAR** – Flood mapping under cloudy/rainy conditions.
2. **Landsat-9** – Optical imagery for crop and land cover classification.
3. **ESA WorldCover** – Built-up area mapping at 10m resolution.
4. **NASA SMAP** – Soil moisture monitoring for surface and root zones.

**Study Area**
Four flood-affected districts in Sindh province were studied deeply in the paper:
- Mehar
- Dadu
- Larkana
- Moro

Note : The code provided below analyses the flood impact over all of Pakistan provincially.



# Study Area & Event Description

The study focuses on four districts in Sindh, Pakistan: **Mehar, Dadu, Larkana, and Moro**, covering ~3,054 km².

**Event Timeline:**
- **Pre-flood period:** May 2021 – May 2022
- **Post-flood period:** July – September 2022

Sindh was the most severely impacted province in the 2022 floods, with:
- Widespread submergence of cropland.
- Significant damage to urban infrastructure.
- Long-term soil saturation affecting future crop cycles.


<a name="methodology"></a>
# Methodology Overview

<p>The workflow follows these main steps:</p>


1. **Pre-flood & Post-flood Imagery Collection**
<ul>
   <li> Sentinel-1 SAR for flood detection.</li>
   <li> Landsat-9 for vegetation and crop classification.</li>
</ul>


2. **SAR Image Processing**
<ul>
   <li> Apply filters to reduce speckle noise.</li>
   <li> Mosaic and clip to Area of Interest (AOI).</li>
   <li> Calculate NDWI to detect water/flooded areas.</li>
</ul>

3. **Land Cover Classification**
<ul>
   <li> Random Forest classification of crops from Landsat-9.</li>
   <li> ESA WorldCover for built-up area mapping.</li>
</ul>

4. **Flood Impact Assessment**
<ul>
   <li> Overlay flood masks with crop and built-up layers.</li>
   <li> Quantify affected areas (sq km).</li>
</ul>


5. **Visualization & Export**
<ul>
   <li> Interactive maps with geemap.</li>
   <li> Export results for further GIS analysis.</li>
</ul>
   

<center>
  <img src="https://github.com/Abubakar-Rana/neurips2025-data-flood-paper/raw/ea1e1ffaab0023e4697bf4b786fb63a4d9a0a283/methodologyLatest.png" width="500">
  <div>Image source: <a href="https://arxiv.org/pdf/2410.07126">FIG-2 from paper</a></div>
</center>



# Hands-On Implementation in Python
<a name="implementation"></a>
<p>In this section, we implement the methodology using Python, Google Earth Engine (GEE), and <code>geemap</code>.</p>
    <h3>Implementation Steps</h3>
    <ol>
      <li>Install dependencies and authenticate GEE.</li>
      <li>Define the Area of Interest (AOI) and time ranges.</li>
      <li>Filter and preprocess Sentinel-1 SAR data.</li>
      <li>Calculate NDWI and detect flooded pixels.</li>
      <li>Classify crops using Landsat-9 with a Random Forest classifier.</li>
      <li>Map built-up areas using ESA WorldCover.</li>
      <li>Visualize results interactively with <code>geemap</code>.</li>
    </ol>
    

This step installs the necessary Python packages in the Colab environment and authenticates Earth Engine. We use geemap for interactive maps in Colab, and the Earth Engine Python API to run the same operations as in the JavaScript script.


In [1]:
!pip uninstall -y earthengine-api geemap ee --quiet
!pip install earthengine-api geemap folium numpy matplotlib --quiet

In [3]:

import ee
import geemap
import folium
ee.Authenticate()
# Replace 'your-project-id' with your actual GEE project ID
ee.Initialize()


<p>Defined here is our Area of Interest(AOI), you can choose an area of your own choice as well, time windows for the before/after images, and SAR processing parameters.</p>

These are experimental parameters:
<ul>
<li>polarization</li>
<li>pass direction</li>
<li>speckle smoothing radius</li>
<li>the difference threshold used to mark flooding.</li>
</ul>

In [ ]:
# # Load AOI from GEE asset
# aoi = ee.FeatureCollection("users/usmannazirbnu/District_Boundary_Pak")

**NOTE**: If the above cell generate error or not accessible uncomment the code of cell below and then run. (and comment the above cell)



In [9]:
aoi = ee.FeatureCollection("FAO/GAUL/2015/level2") \
        .filter(ee.Filter.eq('ADM0_NAME', 'Pakistan'))

In [10]:
# Visual check
Map = geemap.Map(center = [30.3753, 69.3451]
 , zoom=7)
Map.addLayer(aoi, {}, "AOI")
Map


Map(center=[30.3753, 69.3451], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topri…

In [11]:
# Define the time ranges
before_start = '2021-05-01'
before_end = '2021-05-30'
after_start = '2022-07-01'
after_end = '2022-09-30'


<p>Unlike optical sensors such as Landsat, Sentinel-1 SAR does not capture spectral bands in the visible or near-infrared regions needed for the classic NDWI formula:<p>
<p><b>(Green - NIR) / (Green + NIR)</b></p>


<p>Instead, flood detection is performed using <p>SAR backscatter differencing</p>
<ol>
<li> <b>Pre-flood</b> and <b>post-flood</b> SAR mosaics are generated for the Area of Interest (AOI).</li>
<li>The difference in backscatter values (`after - before`) highlights changes in surface water extent.</li>
<li>A <b>threshold</b> is applied to this difference image to produce a binary <b>flood mask</b>, where pixels exceeding the threshold are classified as flooded.</li>
</ol>

<p>This change-detection approach is robust in cloudy or rainy conditions, making it ideal for flood mapping during the monsoon season.<p>


<p>Filter <b>Sentinel-1</b> GRD to the instrument mode, polarization, pass direction, resolution and AOI. Split into before and after collections and print counts and time ranges. This matches the provisioning step in the JavaScript script and serves as quality control before mosaicking.</p>

In [12]:
# Define parameters
polarization = "VV"
pass_direction = "DESCENDING"

# Base collection
collection = ee.ImageCollection("COPERNICUS/S1_GRD") \
    .filter(ee.Filter.eq("instrumentMode", "IW")) \
    .filter(ee.Filter.listContains("transmitterReceiverPolarisation", polarization)) \
    .filter(ee.Filter.eq("orbitProperties_pass", pass_direction)) \
    .filterBounds(aoi) \
    .select(polarization)

# Before and After collections
before_collection = collection.filterDate(before_start, before_end)
after_collection = collection.filterDate(after_start, after_end)

# Count images
before_count = before_collection.size().getInfo()
after_count = after_collection.size().getInfo()

print(f"Before Flood images: {before_count}")
print(f"After Flood images: {after_count}")


Before Flood images: 112
After Flood images: 336


<p><b>Mosaic</b> the filtered collections and apply a focal_mean (moving-window mean) to reduce speckle. The smoothing radius is chosen experimentally; specify in Methods the value used and rationale.</p>

In [13]:
# Create mosaics
before = before_collection.mosaic().clip(aoi)
after = after_collection.mosaic().clip(aoi)

# Apply speckle filter (smoothing)
radius = 50
before_filtered = before.focal_mean(radius, 'circle', 'meters')
after_filtered = after.focal_mean(radius, 'circle', 'meters')

# Visualize
Map = geemap.Map(center = [30.3753, 69.3451], zoom=5)
Map.addLayer(before_filtered, {'min': -25, 'max': 0}, 'Before Flood')
Map.addLayer(after_filtered, {'min': -25, 'max': 0}, 'After Flood')
Map


Map(center=[30.3753, 69.3451], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topri…

<p>Compute a ratio-based difference image (after / before). Pixels with values greater than the difference_threshold are flagged as potential flooded pixels. Then exclude permanent water (JRC seasonality >= 10 months), remove small noisy clumps (connected pixel count), and mask steep slopes (>5°) using a DEM.</p>

In [15]:
# Difference and thresholding
difference = after_filtered.divide(before_filtered)
difference_threshold = 1.12
flood_mask = difference.gt(difference_threshold)

# Remove permanent water
seasonality = ee.Image("JRC/GSW1_0/GlobalSurfaceWater").select("seasonality")
perm_water_mask = seasonality.gte(10)
flood_mask = flood_mask.where(perm_water_mask, 0)

# Remove small objects (connectivity)
flood_mask = flood_mask.updateMask(flood_mask.connectedPixelCount().gte(8))

# Remove slope > 5%
DEM = ee.Image("WWF/HydroSHEDS/03VFDEM")
slope = ee.Algorithms.Terrain(DEM).select("slope")
flood_mask = flood_mask.updateMask(slope.lt(5))

# Visualize result
Map = geemap.Map(center = [30.3753, 69.3451], zoom=8)
Map.addLayer(flood_mask.updateMask(flood_mask), {'palette': ['#FF2F00']}, 'Flooded Areas')
Map


Map(center=[30.3753, 69.3451], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topri…

<p>Calculate area of flooded pixels (<b>m<sup>2</sup></b> → <b>hectares</b>). This cell demonstrates how to print GEE-derived scalar results directly into notebook output for inclusion in the manuscript.</p>

In [16]:
# Calculate area in hectares
pixel_area = flood_mask.multiply(ee.Image.pixelArea())
flood_area_stats = pixel_area.reduceRegion(
    reducer=ee.Reducer.sum(),
    geometry=aoi.geometry(),
    scale=10,
    bestEffort=True
)
flood_area_ha = flood_area_stats.get(polarization).getInfo() / 10000
print(f"Estimated Flooded Area: {round(flood_area_ha)} hectares")


Estimated Flooded Area: 7864921 hectares


<p>Estimate the number of people exposed by overlaying a population dataset (JRC GHSL) with the flood mask. Reproject flood layer to GHSL resolution (250 m) and sum exposed population pixels.</p>

In [17]:

pop = ee.Image('JRC/GHSL/P2016/POP_GPW_GLOBE_V1/2015').clip(aoi)

# Get GHSL projection
ghsl_proj = pop.projection()

# Reproject flood layer to GHSL scale
flooded_res1 = flood_mask.reproject(crs=ghsl_proj)

# Create a raster showing exposed population only using the resampled flood layer
pop_masked = pop.updateMask(flooded_res1).updateMask(pop)

# Calculate total exposed population
pop_stats = pop_masked.reduceRegion(
    reducer=ee.Reducer.sum(),
    geometry=aoi.geometry(),
    scale=250,
    maxPixels=1e9
)

pop_exposed = int(pop_stats.get('population_count').getInfo())
print(f"Exposed Population: {pop_exposed} people")


Exposed Population: 19072725 people


<p>Use <b>MODIS</b> land cover (MCD12Q1 LC_Type1) to isolate cropland and urban classes; reproject the flood mask to MODIS resolution (500 m) and compute affected cropland and urban areas in hectares.</p>

In [18]:
# MODIS Land Cover
LC = ee.ImageCollection('MODIS/061/MCD12Q1') \
    .filterDate('2014-01-01', after_end) \
    .sort('system:index', False) \
    .select("LC_Type1") \
    .first() \
    .clip(aoi)

# Crop mask: Classes 12 (Cropland) and 14 (Cropland/Natural Vegetation)
cropland = LC.updateMask(LC.eq(12).Or(LC.eq(14)))
urban = LC.updateMask(LC.eq(13))

# Reproject flood layer to MODIS scale
flooded_res = flood_mask.reproject(crs=LC.projection())

# Calculate affected cropland using the resampled flood layer
affected_cropland = flooded_res.updateMask(cropland)

affected_urban = flood_mask.updateMask(urban)

# ---- Cropland area calculation ----
crop_stats = affected_cropland.multiply(ee.Image.pixelArea()).reduceRegion(
    reducer=ee.Reducer.sum(),
    geometry=aoi.geometry(),
    scale=500,
    maxPixels=1e9
).getInfo()

crop_area = list(crop_stats.values())[0] / 10000 if crop_stats else 0

# ---- Urban area calculation ----
urban_stats = affected_urban.multiply(ee.Image.pixelArea()).reduceRegion(
    reducer=ee.Reducer.sum(),
    geometry=aoi.geometry(),
    scale=500,
    bestEffort=True
).getInfo()

urban_area = list(urban_stats.values())[0] / 10000 if urban_stats else 0

# Print results
print(f"Affected Cropland: {round(crop_area)} ha")
print(f"Affected Urban Area: {round(urban_area)} ha")

Affected Cropland: 1934524 ha
Affected Urban Area: 43503 ha


<p>Visualize results in an interactive map. geemap renders map tiles inside Colab and supports toggling layers. For publication figures, export PNGs or take map screenshots.</p>

In [19]:

from IPython.display import HTML, display

legend_html = """
<div style="font-family:sans-serif; line-height:1.5;">
  <b style="color:blue;">Legend</b><br>
  <div style="display:flex; align-items:center;">
    <div style="width:20px; height:20px; background-color:#FF0000; margin-right:6px;"></div>
    <span>potentially flooded areas</span>
  </div>
  <div style="display:flex; align-items:center;">
    <div style="width:20px; height:20px; background-color:#00FF00; margin-right:6px;"></div>
    <span>affected cropland</span>
  </div>
  <div style="display:flex; align-items:center;">
    <div style="width:20px; height:20px; background-color:#888888; margin-right:6px;"></div>
    <span>affected urban</span>
  </div>
  <br>
  <b>Exposed population density</b><br>
  &gt; 200
  <div style="width: 100px; height: 10px; background: linear-gradient(to right, yellow, red); margin: 4px 0;"></div>
  0
</div>
"""

display(HTML(legend_html))

Map = geemap.Map(center = [30.3753, 69.3451], zoom=8)

Map.addLayer(before_filtered, {'min': -25, 'max': 0}, 'Before Flood',0)
Map.addLayer(after_filtered, {'min': -25, 'max': 0}, 'After Flood')
Map.addLayer(difference, {'min': 0, 'max': 2}, 'Difference',0)
Map.addLayer(cropland, {'palette': ['green']}, 'Cropland',0)
Map.addLayer(urban, { 'min': 0, 'max': 13.0, 'palette': ['grey']}, 'Urban',0)

Map.addLayer(affected_urban, {'palette': ['purple']}, 'Affected Urban')
# Define cropland visualization
croplandVis = {
    'min': 0,
    'max': 14.0,
    'palette': ['30b21c'],
}
#Map.addLayer(affected_cropland, croplandVis, 'Affected Cropland')
Map.addLayer(flood_mask.updateMask(flood_mask), {'palette': ['#FF2F00']}, 'Flooded Areas')
Map.addLayer(pop_masked, {'min': 0, 'max': 200, 'palette': ['yellow', 'red']}, 'Exposed Population')

Map


Map(center=[30.3753, 69.3451], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topri…

# Results and Discussion
<a name="results-and-discussion"></a>

<center>
  <img src="https://github.com/Abubakar-Rana/neurips2025-data-flood-paper/raw/07e9c38d3fb014dc9ae2e076c73a107d322811c4/advanced-gbt-ajk-wali.png" width="800">
  <br>
  <i>Our tutorial results</i>
</center>


<p>The spatial footprint of the 2022 floods across Pakistan (01-29 August) reveals a massive inundation of nearly 59,000 km<sup>2</sup>, with an estimated 19.7 million people exposed. Out of this, roughly 20,000 km<sup>2</sup> of cropland was affected. The map shows a stark concentration of floodwater in Sindh, where large areas of cultivated land overlapped with flood extent. This explains why Sindh reported devastating agricultural losses during the Kharif season.</p>

<p>In southern Punjab, the flood spread along the Indus basin, again overlapping with cropland and contributing to reduced yields in rice, maize, and cotton. In contrast, Balochistan and Khyber Pakhtunkhwa (KPK) display more scattered flood patches. While cropland loss was relatively smaller here, the floods still caused widespread disruptions to rural communities and infrastructure. The northern regions (Gilgit-Baltistan and Azad Kashmir) saw flooding largely restricted to river valleys, with limited cropland exposure.</p>

<p>The crop classification results from Landsat-9 confirmed that the majority of Kharif crops in Sindh's floodplain were destroyed, particularly rice and cotton fields. By intersecting the Random Forest-derived crop maps with the flood extent, it became clear that Mehar and Larkana districts suffered the heaviest damage.</p>

<p>Furthermore, our results are identical to the results shown by the UNITAR department of UN with the help of UNOSAT team. This further solidifies the usefulness of our code.</p>

<center>
  <img src="https://github.com/Abubakar-Rana/neurips2025-data-flood-paper/raw/07e9c38d3fb014dc9ae2e076c73a107d322811c4/UNOSAT%20Monsoon%20Floods%202022_page-0001.jpg" width="800">
  <br>
  <i>Source: UNITAR–UNOSAT, 2022. Pakistan: Floods – Monsoon 2022</i>
</center>


# References
<a name="references"></a>

<p>Ulloa, N.; Yun, S.-H.; Chiang, S.-H.; and Furuta, R. 2022.
 Sentinel-1 Spatiotemporal Simulation Using Convolutional
 LSTMfor Flood Mapping. Remote Sensing, 14: 246.</p>
 <p>Zhang, L.; and Xia, J. 2021. Flood Detection Using Mul
tiple Chinese Satellite Datasets during 2020 China Summer
 Floods. Remote Sensing, 14: 51</p>
 <p>Haq, M.; Akhtar, M.; Muhammad, S.; Paras, S.; and Rah
matullah, J. 2012. Techniques of remote sensing and GIS
 for flood monitoring and damage assessment: a case study
 of Sindh province, Pakistan. The Egyptian Journal of Re
mote Sensing and Space Science, 15(2): 135-141.</p>
<p>
  Abubakar, H. M.; Khan, A.; Younas, A.; Tahseen, Z.; Arshad, A.; Taj, M.; and Nazir, U. 2024.
  2022 Flood Impact in Pakistan: Remote Sensing Assessment of Agricultural and Urban Damage.
  Available at:
  <a href="https://ojs.aaai.org/index.php/AAAI-SS/article/view/31824" target="_blank" rel="noopener">
    https://ojs.aaai.org/index.php/AAAI-SS/article/view/31824
  </a>
</p>
